# Nemotron 3 Nano — Dr. GRPO RLVR (RTX 6000 Pro v13)

Dr. GRPO (Sea AI Lab, arxiv:2503.20783) with verifiable per-puzzle-type
rewards. Loads the SFT adapter from `nemo-v13-sft-rtx6000.ipynb`.

## Why "Done Right"

Standard GRPO has two biases (Sea AI Lab):
1. **Response-length norm `1/|o_i|`** — favors short correct, long wrong responses
2. **Std normalization** — over-weights easy/hard questions inverse to difficulty variance

Dr. GRPO removes both:
- `loss_type="dr_grpo"` (or fallback `"dapo"`) — replaces length norm with a constant
- `scale_rewards=False` — removes std norm

## Verifiable Rewards (per puzzle type)

Ported from [jhyland01/kaggle_nemotron](https://github.com/jhyland01/kaggle_nemotron):

| Puzzle Type | Verifier Strategy |
|---|---|
| Number Base Conversion | Numeric match in target base |
| Gravitational Constant | Numeric tolerance (rel 1e-2) |
| Unit Conversion | Numeric tolerance (rel 1e-2) |
| Text Encryption | Case-insensitive string match |
| Bit Manipulation | 8-bit binary string match |
| Equation Transformation | String match (numeric or concat) |
| Unknown | Fallback: normalized string + numeric tolerance |

Each completion gets a binary `correct/incorrect` decision against its
known answer. Per-type accuracy logged to TensorBoard.

## Hard rules

- Reward requires `<think>...</think>` **before** `\boxed{}` — blocks naked-boxed reward hacking
- `\boxed{}` extraction is brace-balanced (handles `\boxed{\frac{1}{2}}`)
- Eval is greedy (T=0); training is T=1.0 for exploration — this is expected


In [1]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})

{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0, 'PRETRAINED_ADAPTER_DATASET_PATH': '/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection'}


In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--target", target,
        "--upgrade", "--ignore-installed", wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

Found Triton wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
triton spec: ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7aeb0a7c3320>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

Training environment fixes applied.


In [4]:
import os, sys

# os.environ["PYTHONIOENCODING"] = "utf-8"
# if hasattr(sys.stdout, "reconfigure"):
#     sys.stdout.reconfigure(encoding="utf-8", errors="strict")
# if hasattr(sys.stderr, "reconfigure"):
#     sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# # Repo layout — override via env var if running elsewhere
# REPO_ROOT = os.environ.get("KAGGLE_NEMO_REPO", r"F:/Hackathons/Kaggle-Nemotron")

BASE_MODEL_NAME   = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
SFT_DATA_PATH     =  "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
GRPO_DATA_PATH    =  "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"

OUTPUT_ROOT       = "outputs"
SFT_ADAPTER_DIR   = os.path.join(OUTPUT_ROOT, "sft_adapter")
GRPO_ADAPTER_DIR  = os.path.join(OUTPUT_ROOT, "grpo_adapter")
SUBMISSION_DIR    = os.path.join(OUTPUT_ROOT, "submission_adapter_sft")
TB_LOG_DIR        = os.path.join(OUTPUT_ROOT, "tb_logs_sft")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED          = 42
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print({
    # "REPO_ROOT": REPO_ROOT,
    "BASE_MODEL_NAME": BASE_MODEL_NAME,
    "SFT_DATA_PATH": SFT_DATA_PATH,
    "GRPO_DATA_PATH": GRPO_DATA_PATH,
    "SFT_ADAPTER_DIR": SFT_ADAPTER_DIR,
    "GRPO_ADAPTER_DIR": GRPO_ADAPTER_DIR,
})


{'BASE_MODEL_NAME': 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16', 'SFT_DATA_PATH': '/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv', 'GRPO_DATA_PATH': '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv', 'SFT_ADAPTER_DIR': 'outputs/sft_adapter', 'GRPO_ADAPTER_DIR': 'outputs/grpo_adapter'}


In [5]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers",
            "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels): return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")

    print("Offline package installation finished.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Offline package installation finished.


In [6]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"  # SFT uses right-padding for causal LM loss
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-23 09:16:58.564128: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779527818.761724      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779527818.824742      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779527819.337954      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779527819.337966      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779527819.337967      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded with Unsloth.


## Path & Run Configuration


In [7]:
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
import re
from collections import Counter

# Inventory linear modules — for fallback path + sanity print
linear_modules = []
for name, mod in model.named_modules():
    if mod.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
        linear_modules.append(name)

if os.path.isdir(SFT_ADAPTER_DIR) and os.path.exists(os.path.join(SFT_ADAPTER_DIR, "adapter_config.json")):
    print(f"Loading SFT adapter from {SFT_ADAPTER_DIR} (is_trainable=True)")
    model = PeftModel.from_pretrained(model, SFT_ADAPTER_DIR, is_trainable=True)
    # Re-enable gradient checkpointing on the wrapped model
    model.gradient_checkpointing_enable()
    model.print_trainable_parameters()
else:
    print(f"[WARN] No SFT adapter at {SFT_ADAPTER_DIR}.  Cold-start GRPO with fresh LoRA.")
    target_regex = (
        r".*("
        r"self_attn\.(q|k|v|o)_proj"
        r"|mamba\.(in|out|x|dt)_proj"
        r"|shared_expert\.(gate|up|down)_proj"
        r")$"
    )
    matched = [n for n in linear_modules if re.match(target_regex, n)]
    target_modules = target_regex if matched else [
        "q_proj","k_proj","v_proj","o_proj","in_proj","out_proj",
        "x_proj","dt_proj","gate_proj","up_proj","down_proj",
    ]
    lora_config = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.0, bias="none",
        target_modules=target_modules, task_type=TaskType.CAUSAL_LM, use_rslora=True,
    )
    model = get_peft_model(model, lora_config)
    model.gradient_checkpointing_enable()
    model.print_trainable_parameters()


[WARN] No SFT adapter at outputs/sft_adapter.  Cold-start GRPO with fresh LoRA.
trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


## Verifiable Reward Functions (per-type)

Implements the deterministic verifiers from the reference repo. Each
completion's `\boxed{}` answer is compared against the labeled ground truth
using a type-aware comparator (numeric tolerance for gravity / unit conversion,
binary normalization for bit manipulation, etc.).


In [8]:
import re

# ---------- Puzzle type classifier (regex from jhyland01/kaggle_nemotron) ----------
_PUZZLE_PATTERNS = [
    ("Number Base Conversion", re.compile(r"numeral system|base[- ]?\d|number.*convert|radix|secret number", re.IGNORECASE)),
    ("Gravitational Constant", re.compile(r"gravit|gravity|falling|free.?fall|acceleration due to", re.IGNORECASE)),
    ("Equation Transformation", re.compile(r"transformation rule|equation.*transform|secret.*rule.*equation|rule.*applied.*equation", re.IGNORECASE)),
    ("Text Encryption",         re.compile(r"encrypt|cipher|secret.*code.*letter|coded.*message|secret.*text", re.IGNORECASE)),
    ("Bit Manipulation",        re.compile(r"bit.?manipul|binary|8.?bit|bitwise|bit.*transform", re.IGNORECASE)),
    ("Unit Conversion",         re.compile(r"unit.?conver|measurement|becomes.*\d|secret.*conver.*measur", re.IGNORECASE)),
]

def classify_puzzle(prompt: str) -> str:
    for label, pat in _PUZZLE_PATTERNS:
        if pat.search(prompt or ""):
            return label
    return "Unknown"

# ---------- Boxed extraction (brace-balanced) ----------
_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def _content(c):
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return c[-1].get("content", "")
    return str(c)

def extract_boxed(text):
    idx = text.find("\\boxed{")
    if idx == -1:
        m = _BOXED_RE.search(text)
        return m.group(1).strip() if m else None
    depth, start = 1, idx + 7
    for i in range(start, len(text)):
        if text[i] == "{":   depth += 1
        elif text[i] == "}": depth -= 1
        if depth == 0:       return text[start:i].strip()
    return text[start:].strip()

def _norm(s):
    return str(s).strip().lower().replace(" ", "").replace(",", "")

# ---------- Type-aware verifier ----------
def _numeric_match(predicted, expected, rel_tol=1e-2, abs_tol=1e-4):
    try:
        pf = float(str(predicted).replace(",", "").strip())
        ef = float(str(expected).replace(",", "").strip())
    except (ValueError, TypeError):
        return False
    return abs(pf - ef) <= max(rel_tol * max(1.0, abs(ef)), abs_tol)

def _integer_match(predicted, expected, bases=(2,8,10,16)):
    """
    Try to interpret both strings as integers in any of the given bases.
    Returns True if there exist bases b1, b2 such that int(p, b1) == int(e, b2).
    """
    p_str = _norm(predicted)
    e_str = _norm(expected)

    def parse(s, base):
        try:
            # Remove common prefixes (0b, 0x) if they match the base
            if base == 16 and s.startswith("0x"):
                s = s[2:]
            elif base == 2 and s.startswith("0b"):
                s = s[2:]
            return int(s, base)
        except ValueError:
            return None

    for b1 in bases:
        p_val = parse(p_str, b1)
        if p_val is None:
            continue
        for b2 in bases:
            e_val = parse(e_str, b2)
            if e_val is None:
                continue
            if p_val == e_val:
                return True
    return False

def verify_answer(predicted, expected, puzzle_type):
    if predicted is None:
        return False
    pn, en = _norm(predicted), _norm(expected)
    # Always accept exact normalized string match
    if pn == en:
        return True

    if puzzle_type in ("Gravitational Constant", "Unit Conversion"):
        return _numeric_match(predicted, expected, rel_tol=1e-2, abs_tol=1e-4)
    if puzzle_type in ("Bit Manipulation", "Number Base Conversion"):
        return _integer_match(predicted, expected)
    if puzzle_type == "Text Encryption":
        # case-insensitive exact, plus whitespace-insensitive
        return pn.replace("'", "") == en.replace("'", "")
    if puzzle_type == "Equation Transformation":
        return _numeric_match(predicted, expected) or pn == en
    # Unknown / fallback: numeric tolerance + string match
    return _numeric_match(predicted, expected)

# ---------- Smoke tests ----------
assert verify_answer("42", "42", "Unknown")
assert verify_answer("9.81", "9.80", "Gravitational Constant")          # within 1e-2 rel
assert not verify_answer("9.81", "10.0", "Gravitational Constant")
assert verify_answer("00101010", "00101010", "Bit Manipulation")
assert verify_answer("00101010", "42", "Bit Manipulation")              # bin vs dec
assert verify_answer("0xFF", "255", "Number Base Conversion")
assert verify_answer("HELLO", "hello", "Text Encryption")
print("Verifier smoke tests pass.")

Verifier smoke tests pass.


## Reward Functions

Reward components (summed):

1. **Format** — `+0.5` for any `\boxed{}`, `+0.3` for `<think>...</think>`
   appearing **before** the boxed answer
2. **Verifiable accuracy** — `+2.0` if type-aware verifier says correct, else `0.0`
3. **DAPO overlong penalty** — soft `[0, -1]` between
   `max_completion + buffer` and `2x max_completion`

Per-type accuracy logged to TensorBoard via `log_metric`.


In [9]:
GRPO_MAX_COMPLETION = 3000   # Dr. GRPO recipe (Table 6 of arxiv:2503.20783)
GRPO_MAX_PROMPT     = 3072
OVERLONG_BUFFER     = 512

# ---------- Format reward ----------
def format_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        text = _content(c)
        has_boxed = bool(_BOXED_RE.search(text))
        has_think = bool(_THINK_RE.search(text))
        think_before_boxed = (
            has_think and has_boxed and
            text.find("</think>") < text.rfind("\\boxed{")
        )
        r = 0.0
        if has_boxed:           r += 0.5
        if think_before_boxed:  r += 0.3
        rewards.append(r)
    return rewards

# ---------- Verifiable accuracy reward (binary, type-aware) ----------
# Track per-type accuracy for logging
_per_type_stats = {}

def accuracy_reward(completions, answer, prompt=None, **kwargs):
    # 'prompt' column may or may not be passed depending on trainer state
    prompts_iter = prompt if prompt is not None else [None] * len(completions)
    rewards = []
    for c, expected, raw_prompt in zip(completions, answer, prompts_iter):
        text = _content(c)
        predicted = extract_boxed(text)
        # Classify (uses raw user prompt if available, else completion itself)
        ptype = classify_puzzle(raw_prompt or text)
        ok = verify_answer(predicted, expected, ptype)
        rewards.append(2.0 if ok else 0.0)

        # Track per-type stats
        s = _per_type_stats.setdefault(ptype, {"n": 0, "correct": 0})
        s["n"] += 1
        if ok:
            s["correct"] += 1
    return rewards

# ---------- DAPO overlong shaping ----------
def overlong_penalty(completions, **kwargs):
    rewards = []
    soft = GRPO_MAX_COMPLETION + OVERLONG_BUFFER
    hard = GRPO_MAX_COMPLETION * 2
    for c in completions:
        text = _content(c)
        n = len(text) / 4.0   # rough char->token estimate
        if   n <= soft: r = 0.0
        elif n >= hard: r = -1.0
        else:           r = -((n - soft) / (hard - soft))
        rewards.append(r)
    return rewards

def combined_reward(completions, answer, prompt=None, **kwargs):
    f = format_reward(completions)
    a = accuracy_reward(completions, answer=answer, prompt=prompt)
    o = overlong_penalty(completions)
    return [x + y + z for x, y, z in zip(f, a, o)]

# Smoke test
test_compl = ["<think>\nthink\n</think>\n\\boxed{42}",
              "no think tag here \\boxed{42}",
              "<think>\nblah\n</think>\n\\boxed{wrong}"]
test_ans = ["42", "42", "42"]
test_prompts = ["compute the result", "compute the result", "compute the result"]
print("Smoke combined_reward:", combined_reward(test_compl, answer=test_ans, prompt=test_prompts))
print("Per-type stats:", _per_type_stats)
_per_type_stats.clear()
print("Reward functions ready.")


Smoke combined_reward: [2.8, 2.5, 0.8]
Per-type stats: {'Unknown': {'n': 3, 'correct': 2}}
Reward functions ready.


## GRPO Dataset Prep — Raw competition `train.csv`

Keeps **prompt + answer** only. The model generates rollouts; the verifier
scores them. No pre-computed CoT.


In [10]:
import pandas as pd
from datasets import Dataset as HFDataset

df_grpo = pd.read_csv(GRPO_DATA_PATH)
print(f"GRPO data: {len(df_grpo)} rows.  Columns: {list(df_grpo.columns)}")
if not {"prompt", "answer"}.issubset(df_grpo.columns):
    raise ValueError(f"train.csv must have 'prompt' and 'answer'; got {list(df_grpo.columns)}")

df_grpo = df_grpo.dropna(subset=["prompt", "answer"]).reset_index(drop=True)
df_grpo = df_grpo.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Sample type distribution before training
type_dist = df_grpo["prompt"].sample(min(1000, len(df_grpo)), random_state=0).apply(classify_puzzle).value_counts()
print("\nSample puzzle type distribution (first 1000 prompts):")
print(type_dist)

records_g = []
for _, row in df_grpo.iterrows():
    raw_prompt = str(row["prompt"])
    msgs = [{"role": "user", "content": raw_prompt + PROMPT_SUFFIX}]
    try:
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                    add_generation_prompt=True,
                                                    enable_thinking=True)
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False,
                                                    add_generation_prompt=True)
    records_g.append({
        "prompt": prompt_text,
        "answer": str(row["answer"]),
        # NOTE: keeping the raw user content in a separate column lets the reward
        # fn classify by puzzle type without re-stripping the chat template.
        "raw_prompt": raw_prompt,
    })

grpo_dataset = HFDataset.from_list(records_g)
print(f"\nGRPO records: {len(records_g)}")


GRPO data: 9500 rows.  Columns: ['id', 'prompt', 'answer']

Sample puzzle type distribution (first 1000 prompts):
prompt
Gravitational Constant     181
Bit Manipulation           175
Unit Conversion            173
Number Base Conversion     160
Equation Transformation    156
Text Encryption            155
Name: count, dtype: int64

GRPO records: 9500


## Per-Type Accuracy Logging Callback

Wraps the trainer log step so per-type accuracy snapshots get pushed to
TensorBoard each logging interval. Tracks moving accuracy per puzzle type.


In [11]:
from transformers import TrainerCallback

class PerTypeAccuracyCallback(TrainerCallback):
    def __init__(self, stats_dict):
        super().__init__()
        self.stats = stats_dict   # references _per_type_stats from reward cell

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        for ptype, s in self.stats.items():
            if s["n"] > 0:
                logs[f"acc/{ptype.replace(' ', '_').lower()}"] = s["correct"] / s["n"]
                logs[f"count/{ptype.replace(' ', '_').lower()}"] = s["n"]
        # Reset window every log step so we see per-window accuracy, not cumulative
        for s in self.stats.values():
            s["n"] = 0
            s["correct"] = 0


## Dr. GRPO Trainer Config

Strict Dr. GRPO recipe from WINNING_PLAN.md Phase 1 + DAPO upgrades from
Phase 2 where TRL supports them.

| Param | Value | Why |
|---|---|---|
| `learning_rate` | 1e-6 | Dr. GRPO recipe (constant) |
| `lr_scheduler_type` | constant | Dr. GRPO recipe |
| `num_generations` | 8 | Dr. GRPO minimum |
| `temperature` | 1.0 | exploration during RL |
| `top_p` | 1.0 | no truncation |
| `beta` | 0.0 | no KL, no reference model — saves ~50% mem |
| `max_prompt_length` | 3072 | accommodates long puzzle prompts |
| `max_completion_length` | 3000 | Dr. GRPO recipe |
| `per_device_train_batch_size` | 1 | full group fits |
| `gradient_accumulation_steps` | 16 | effective batch = 16, divisible by num_gen=8 |
| `max_grad_norm` | 1.0 | not 0.1 (too tight) |
| `loss_type` | `dr_grpo` (or `dapo` fallback) | drop length norm |
| `scale_rewards` | False | drop std norm |
| `mask_truncated_completions` | True | exclude truncated from loss |
| `epsilon_high` | 0.28 | DAPO clip-higher |


In [ ]:
import os, sys, inspect, gc, time

# ---------------------------------------------------------------------------
# CRASH FIX: Unsloth's compiled GRPO trainer is INCOMPATIBLE with Nemotron.
#   .../UnslothGRPOTrainer.py  chunked_hidden_states_selective_log_softmax
#     chunk_logits = chunk_hidden_states @ lm_head.t()
#   RuntimeError: mat1 and mat2 shapes cannot be multiplied (750x131072 and 2688x131072)
#
#   Unsloth assumes the model returns last_hidden_state (dim=hidden=2688) and
#   applies lm_head itself. Nemotron-H returns LOGITS already (dim=vocab=131072),
#   so Unsloth projects a second time -> shape clash. This is a logic bug, not a
#   compile-trace guard, so disabling Dynamo alone does NOT fix it.
#
# Fix: bypass Unsloth's patched trainer entirely. Purge unsloth + trl from
# sys.modules (and unsloth import hooks from sys.meta_path) and re-import trl
# CLEAN so `GRPOTrainer` is vanilla TRL, whose compute_loss uses plain
# `model(input_ids).logits` (no chunked hidden trick). The already-loaded
# `model` object stays valid (its bound methods are unaffected by the purge).
# ---------------------------------------------------------------------------
os.environ["TORCHDYNAMO_DISABLE"]   = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch
import torch._dynamo
torch._dynamo.config.disable = True
torch._dynamo.reset()

# Drop unsloth + trl from the module cache.
for _m in list(sys.modules):
    if _m == "trl" or _m.startswith("trl.") or "unsloth" in _m.lower():
        del sys.modules[_m]

# Remove Unsloth import hooks so re-importing trl does NOT re-trigger the patch.
sys.meta_path = [f for f in sys.meta_path
                 if "unsloth" not in type(f).__module__.lower()]

from trl import GRPOTrainer, GRPOConfig
assert "unsloth" not in GRPOTrainer.__module__.lower(), \
    f"Still using Unsloth trainer: {GRPOTrainer.__module__}"
print(f"GRPOTrainer module: {GRPOTrainer.__module__}  (vanilla TRL, dynamo disabled)")

_cfg_params = set(inspect.signature(GRPOConfig.__init__).parameters.keys())
HAS_SCALE_REWARDS = "scale_rewards" in _cfg_params
HAS_LOSS_TYPE     = "loss_type"     in _cfg_params
HAS_MASK_TRUNC    = "mask_truncated_completions" in _cfg_params
HAS_EPSILON_HIGH  = "epsilon_high"  in _cfg_params

print(f"TRL flags: scale_rewards={HAS_SCALE_REWARDS} loss_type={HAS_LOSS_TYPE} "
      f"mask_trunc={HAS_MASK_TRUNC} eps_high={HAS_EPSILON_HIGH}")

grpo_kwargs = dict(
    output_dir                   = os.path.join(OUTPUT_ROOT, "grpo_run"),
    num_train_epochs             = 1,
    per_device_train_batch_size  = 1,
    gradient_accumulation_steps  = 4,        # effective batch = 16
    learning_rate                = 1e-6,      # Dr. GRPO recipe
    lr_scheduler_type            = "constant",
    warmup_ratio                 = 0.0,
    max_prompt_length            = GRPO_MAX_PROMPT,
    max_completion_length        = GRPO_MAX_COMPLETION,
    num_generations              = 4,
    temperature                  = 1.0,
    top_p                        = 1.0,
    beta                         = 0.0,
    optim                        = "paged_adamw_8bit",
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.999,
    weight_decay                 = 0.0,        # Dr. GRPO: no weight decay
    max_grad_norm                = 1.0,
    logging_steps                = 1,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "tensorboard",
    save_strategy                = "no",
    bf16                         = True,
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": True},   # Mamba-safe (CLAUDE.md)
    seed                         = SEED,
    remove_unused_columns        = False,
)

# Apply Dr. GRPO / DAPO knobs where TRL supports them
if HAS_SCALE_REWARDS:
    grpo_kwargs["scale_rewards"] = False                  # Dr. GRPO: drop std norm
if HAS_LOSS_TYPE:
    # Prefer 'dr_grpo' if available, fall back to 'dapo' (both remove length bias)
    try:
        GRPOConfig(loss_type="dr_grpo")
        grpo_kwargs["loss_type"] = "dr_grpo"
    except Exception:
        grpo_kwargs["loss_type"] = "dapo"
if HAS_MASK_TRUNC:
    grpo_kwargs["mask_truncated_completions"] = True
if HAS_EPSILON_HIGH:
    grpo_kwargs["epsilon"]      = 0.2
    grpo_kwargs["epsilon_high"] = 0.28                    # DAPO clip-higher

grpo_config = GRPOConfig(**grpo_kwargs)

print("\n" + "=" * 60)
print("  Dr. GRPO RLVR CONFIG (RTX 6000 Pro v13)")
print("=" * 60)
print(f"  LR (const):     {grpo_config.learning_rate}")
print(f"  Num gen:        {grpo_config.num_generations}")
print(f"  Beta (KL):      {grpo_config.beta}")
print(f"  Max comp len:   {grpo_config.max_completion_length}")
print(f"  Temperature:    {grpo_config.temperature}")
print(f"  Batch:          {grpo_config.per_device_train_batch_size} x {grpo_config.gradient_accumulation_steps} = "
      f"{grpo_config.per_device_train_batch_size*grpo_config.gradient_accumulation_steps} eff")
print(f"  scale_rewards:  {getattr(grpo_config, 'scale_rewards', 'n/a')}")
print(f"  loss_type:      {getattr(grpo_config, 'loss_type', 'n/a')}")
print(f"  mask_trunc:     {getattr(grpo_config, 'mask_truncated_completions', 'n/a')}")
print(f"  epsilon_high:   {getattr(grpo_config, 'epsilon_high', 'n/a')}")
print("=" * 60 + "\n")

## Launch GRPO Training


In [13]:
import gc, time, torch

# The combined reward needs `prompt` access for puzzle-type classification.
# We pass raw prompts via the dataset column 'raw_prompt'; TRL forwards
# extra dataset columns to the reward fn via **kwargs. Wrap to inject.

def combined_reward_with_prompt(completions, answer, raw_prompt=None, **kwargs):
    return combined_reward(completions, answer=answer, prompt=raw_prompt)

grpo_trainer = GRPOTrainer(
    model            = model,
    args             = grpo_config,
    train_dataset    = grpo_dataset,
    reward_funcs     = [combined_reward_with_prompt],
    processing_class = tokenizer,
    callbacks        = [PerTypeAccuracyCallback(_per_type_stats)],
)

torch.cuda.empty_cache(); gc.collect()

print("Starting Dr. GRPO RLVR training...")
t0 = time.time()
grpo_trainer.train()
print(f"Dr. GRPO done in {(time.time()-t0)/60:.1f} min")

os.makedirs(GRPO_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)
print(f"GRPO adapter saved -> {GRPO_ADAPTER_DIR}")

Starting Dr. GRPO RLVR training...


`generation_config` default values have been modified to match model-specific defaults: {'max_length': 262144}. If this is not desired, please set these values explicitly.
[transformers_modules._1.modeling_nemotron_h|WARNING]NemotronH requires an initialized `NemotronHHybridDynamicCache` to return a cache. None was provided, so no cache will be returned.


Unsloth: Will smartly offload gradients to save VRAM!


TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <built-in function matmul>(*(FakeTensor(..., device='cuda:0', size=(((s87 + 3)//4), s12),
           dtype=torch.bfloat16, grad_fn=<ToCopyBackward0>), FakeTensor(..., device='cuda:0', size=(2688, 131072), dtype=torch.bfloat16)), **{}): got RuntimeError('a and b must have same reduction dim, but got [((s87 + 3)//4), s12] X [2688, 131072].')

from user code:
   File "/kaggle/working/unsloth_compiled_cache/UnslothGRPOTrainer.py", line 136, in chunked_hidden_states_selective_log_softmax
    chunk_logits = chunk_hidden_states.to(lm_head.dtype) @ lm_head.t()

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


## Package `submission.zip`

Copies adapter files from `GRPO_ADAPTER_DIR`, patches
`inference_mode=True`, `lora_dropout=0.0`, and base model name. Final
artifact: `outputs/rtx6000_v13/submission.zip`.


In [ ]:
import json, shutil, zipfile

os.makedirs(SUBMISSION_DIR, exist_ok=True)
required = ["adapter_config.json", "adapter_model.safetensors"]
for fname in required:
    sp = os.path.join(GRPO_ADAPTER_DIR, fname)
    dp = os.path.join(SUBMISSION_DIR, fname)
    if not os.path.exists(sp):
        raise FileNotFoundError(f"Missing: {sp}")
    shutil.copy2(sp, dp)
    print(f"  copied {fname}  ({os.path.getsize(dp)/1024/1024:.1f} MB)")

cfg_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_ROOT, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required:
        zf.write(os.path.join(SUBMISSION_DIR, fname), fname)
print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB — ready.")


## Notes

- **TensorBoard**: `tensorboard --logdir outputs/rtx6000_v13/tb_logs_grpo`
- **Per-type accuracy**: shows up as `acc/<puzzle_type>` metrics
- **Watch metrics**:
  - `reward` — should rise (overall learning)
  - `completions/clipped_ratio` — should drop (less truncation)
  - `frac_reward_zero_std` — should drop (more learning signal per group)
  - `acc/bit_manipulation`, `acc/text_encryption`, etc. — per-type progress
- **OOM fallback**: drop `num_generations` 8→4, `max_completion_length` 3000→1536, or `gradient_accumulation_steps` 16→8
- **Cold-start fallback**: if SFT adapter is missing, the load cell falls back to a fresh LoRA — slower but works
- **vLLM smoke test**: load the resulting `submission.zip` through a local vLLM + LoRA inference test before uploading (vLLM+LoRA+Mamba is fragile)
